In [285]:
#Data Science Final Project - Abdulsamad Sulyman (AKS2288)
#This notebook is dedicated to machine learning modeling and future climate–yield forecasting.

#IMPORTS
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings

In [291]:
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 11

output_dir = "/Users/samadsulyman/Desktop/Data Science Project/Plots - ML_Forecasts"
os.makedirs(output_dir, exist_ok=True)
print(f"Saving ML forecast plots to: {output_dir}")

# =============================
# Load Historical Data
# =============================
print("\n" + "="*60)
print("Loading historical datasets...")
print("="*60)

df1 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset1.csv")
df2 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset2.csv")
df3 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset3.csv")
df4 = pd.read_csv("/Users/samadsulyman/Desktop/Data Science Project/Processed Data/ProcessedDataset4.csv")

df1["date"] = pd.to_datetime(df1["date"], errors="coerce")
df2["date"] = pd.to_datetime(df2["date"], errors="coerce")
df4["date"] = pd.to_datetime(df4["date"], errors="coerce")

df1["Temp_C"] = df1["AverageTemperature"]
df2["Temp_C"] = df2["AvgTemperature"]

# Convert numeric columns
for col in ['avg_temp_c', 'precipitation_mm', 'avg_wind_speed_kmh']:
    df4[col] = pd.to_numeric(df4[col], errors='coerce')

print("Datasets loaded")

# =============================
# City Information
# =============================
city_coords = {
    'Abidjan': (5.3600, -4.0083),
    'Accra': (5.6037, -0.1870),
    'Addis Ababa': (9.0320, 38.7469),
    'Addis Abeba': (9.0320, 38.7469),
    'Algiers': (36.7538, 3.0588),
    'Cairo': (30.0444, 31.2357),
    'Dakar': (14.7167, -17.4677),
    'Dar Es Salaam': (-6.7924, 39.2083),
    'Johannesburg': (-26.2041, 28.0473),
    'Kampala': (0.3476, 32.5825),
    'Khartoum': (15.5007, 32.5599),
    'Kinshasa': (-4.4419, 15.2663),
    'Lagos': (6.5244, 3.3792),
    'Luanda': (-8.8383, 13.2344),
    'Maputo': (-25.9692, 32.5732),
    'Nairobi': (-1.2864, 36.8172)
}

city_to_country = {
    'Abidjan': "Côte d'Ivoire",
    'Accra': 'Ghana',
    'Addis Ababa': 'Ethiopia',
    'Addis Abeba': 'Ethiopia',
    'Algiers': 'Algeria',
    'Cairo': 'Egypt',
    'Dakar': 'Senegal',
    'Dar Es Salaam': 'United Republic of Tanzania',
    'Johannesburg': 'South Africa',
    'Kampala': 'Uganda',
    'Khartoum': 'Sudan',
    'Kinshasa': 'Democratic Republic of the Congo',
    'Lagos': 'Nigeria',
    'Luanda': 'Angola',
    'Maputo': 'Mozambique',
    'Nairobi': 'Kenya'
}

# =============================
# Prepare Training Data
# =============================
print("\n" + "="*60)
print("Preparing training data...")
print("="*60)

def prepare_historical_features(df, city, start_year=1960, end_year=2020):
    """Extract features from historical temperature data"""
    
    city_data = df[(df['City'] == city) & 
                   (df['date'].dt.year >= start_year) & 
                   (df['date'].dt.year <= end_year)].copy()
    
    if len(city_data) < 50:
        return None
    
    city_data['year'] = city_data['date'].dt.year
    city_data['decade'] = (city_data['year'] // 10) * 10
    
    # Create features by decade
    decade_features = []
    
    for decade in sorted(city_data['decade'].unique()):
        decade_data = city_data[city_data['decade'] == decade]
        
        if len(decade_data) < 10:
            continue
        
        # Temperature statistics
        mean_temp = decade_data['Temp_C'].mean()
        max_temp = decade_data['Temp_C'].max()
        min_temp = decade_data['Temp_C'].min()
        std_temp = decade_data['Temp_C'].std()
        
        # Calculate trend within decade
        yearly_avg = decade_data.groupby('year')['Temp_C'].mean()
        if len(yearly_avg) >= 3:
            years = yearly_avg.index.values
            temps = yearly_avg.values
            slope, _, _, _, _ = stats.linregress(years, temps)
            decade_trend = slope * 10
        else:
            decade_trend = 0
        
        # Geographic features
        lat, lon = city_coords.get(city, (0, 0))
        
        decade_features.append({
            'City': city,
            'Decade': decade,
            'Mean_Temp': mean_temp,
            'Max_Temp': max_temp,
            'Min_Temp': min_temp,
            'Std_Temp': std_temp,
            'Decade_Trend': decade_trend,
            'Latitude': lat,
            'Longitude': lon,
            'Abs_Latitude': abs(lat),  # Distance from equator
            'Years_From_1960': (decade - 1960) / 10  # Normalized time feature
        })
    
    return pd.DataFrame(decade_features)

# Collect all historical data
all_training_data = []

for city in city_coords.keys():
    print(f"  Processing {city}...")
    
    # Try df1 first
    city_features = prepare_historical_features(df1, city)
    
    # If not enough data, try df2
    if city_features is None or len(city_features) < 3:
        city_features = prepare_historical_features(df2, city, 1995, 2020)
    
    if city_features is not None:
        all_training_data.append(city_features)

training_df = pd.concat(all_training_data, ignore_index=True)
print(f"\nPrepared {len(training_df)} training samples from {training_df['City'].nunique()} cities")

# =============================
# Train Machine Learning Models
# =============================
print("\n" + "="*60)
print("Training ML models...")
print("="*60)

# Prepare features and target
feature_cols = ['Mean_Temp', 'Max_Temp', 'Min_Temp', 'Std_Temp', 'Decade_Trend',
                'Latitude', 'Longitude', 'Abs_Latitude', 'Years_From_1960']

X = training_df[feature_cols].values
y = training_df['Mean_Temp'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train ensemble model
print("  Training Random Forest...")
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)
rf_score = rf_model.score(X_test_scaled, y_test)

print("  Training Gradient Boosting...")
gb_model = GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_score = gb_model.score(X_test_scaled, y_test)

print(f"\nModel Performance (R² score):")
print(f"  Random Forest: {rf_score:.3f}")
print(f"  Gradient Boosting: {gb_score:.3f}")

# Use ensemble average for predictions
def ensemble_predict(X):
    rf_pred = rf_model.predict(X)
    gb_pred = gb_model.predict(X)
    return (rf_pred + gb_pred) / 2

# =============================
# Generate Forecasts for 2030, 2040, 2050
# =============================
print("\n" + "="*60)
print("Generating forecasts for 2030, 2040, 2050...")
print("="*60)

forecast_years = [2030, 2040, 2050]
all_forecasts = []

for city in city_coords.keys():
    # Get most recent historical data for this city
    city_historical = training_df[training_df['City'] == city]
    
    if len(city_historical) == 0:
        continue
    
    # Use most recent decade as baseline
    latest_data = city_historical.iloc[-1]
    
    lat, lon = city_coords[city]
    
    for year in forecast_years:
        # Calculate expected warming based on IPCC scenarios
        # Conservative: 0.2°C per decade
        # Moderate: 0.3°C per decade  
        # High: 0.4°C per decade
        
        decades_ahead = (year - 2020) / 10
        
        for scenario, warming_rate in [('Conservative', 0.2), ('Moderate', 0.3), ('High', 0.4)]:
            # Project features
            projected_mean = latest_data['Mean_Temp'] + (warming_rate * decades_ahead)
            projected_max = latest_data['Max_Temp'] + (warming_rate * decades_ahead * 1.2)  # Extremes increase faster
            projected_min = latest_data['Min_Temp'] + (warming_rate * decades_ahead * 0.8)
            projected_std = latest_data['Std_Temp'] * (1 + decades_ahead * 0.05)  # Variability increases
            projected_trend = latest_data['Decade_Trend'] * 1.1  # Trend accelerates
            
            # Create feature vector
            X_future = np.array([[
                projected_mean,
                projected_max,
                projected_min,
                projected_std,
                projected_trend,
                lat,
                lon,
                abs(lat),
                (year - 1960) / 10
            ]])
            
            # Scale and predict
            X_future_scaled = scaler.transform(X_future)
            predicted_temp = ensemble_predict(X_future_scaled)[0]
            
            # Calculate risk score (0-100)
            # Based on: absolute temperature, change from baseline, extremes
            baseline_temp = latest_data['Mean_Temp']
            temp_increase = predicted_temp - baseline_temp
            
            risk_score = (
                (predicted_temp / 35) * 30 +  # Absolute temperature
                (temp_increase / 3) * 40 +  # Temperature increase
                (projected_std / 10) * 30  # Variability
            )
            risk_score = min(100, max(0, risk_score))
            
            # Classify risk level
            if risk_score < 40:
                risk_level = 'Low'
            elif risk_score < 60:
                risk_level = 'Moderate'
            elif risk_score < 80:
                risk_level = 'High'
            else:
                risk_level = 'Extreme'
            
            all_forecasts.append({
                'City': city,
                'Year': year,
                'Scenario': scenario,
                'Predicted_Temp_C': predicted_temp,
                'Baseline_Temp_C': baseline_temp,
                'Temp_Increase_C': temp_increase,
                'Risk_Score': risk_score,
                'Risk_Level': risk_level,
                'Latitude': lat,
                'Longitude': lon
            })

forecast_df = pd.DataFrame(all_forecasts)
print(f"Generated {len(forecast_df)} forecast scenarios")

# =============================
# PLOT 1: Forecast Maps for 2030, 2040, 2050 (Moderate Scenario)
# =============================
print("\n" + "="*60)
print("Creating forecast maps...")
print("="*60)

moderate_forecasts = forecast_df[forecast_df['Scenario'] == 'Moderate']

fig, axes = plt.subplots(1, 3, figsize=(22, 8))

colors = ['#2ECC40', '#FFDC00', '#FF851B', '#FF4136', '#85144b']
cmap = LinearSegmentedColormap.from_list('risk', colors, N=100)

for idx, year in enumerate(forecast_years):
    ax = axes[idx]
    
    year_data = moderate_forecasts[moderate_forecasts['Year'] == year]
    
    ax.set_xlim(-20, 55)
    ax.set_ylim(-35, 40)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    
    scatter = ax.scatter(year_data['Longitude'], year_data['Latitude'],
                        s=year_data['Risk_Score'] * 8,
                        c=year_data['Risk_Score'],
                        cmap=cmap,
                        alpha=0.75,
                        edgecolor='black',
                        linewidth=2.5,
                        vmin=0, vmax=100,
                        zorder=5)
    
    for _, row in year_data.iterrows():
        ax.annotate(f"{row['City']}\n{row['Risk_Score']:.0f}\n+{row['Temp_Increase_C']:.1f}°C",
                   (row['Longitude'], row['Latitude']),
                   fontsize=8, weight='bold',
                   ha='center', va='bottom',
                   xytext=(0, 10), textcoords='offset points',
                   bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                            alpha=0.85, edgecolor='black', linewidth=1.5))
    
    ax.set_xlabel('Longitude', fontsize=11, weight='bold')
    ax.set_ylabel('Latitude', fontsize=11, weight='bold')
    ax.set_title(f'{year} Projection', fontsize=14, weight='bold', pad=10)

cbar = fig.colorbar(scatter, ax=axes, orientation='horizontal',
                   fraction=0.05, pad=0.08, aspect=40)
cbar.set_label('Climate Risk Score (0-100)', fontsize=13, weight='bold')
cbar.ax.tick_params(labelsize=11)

fig.suptitle('Machine Learning Climate Risk Forecasts (Moderate Scenario)\nHigh-Risk Zone Projections', 
            fontsize=18, weight='bold', y=0.98)

legend_text = ("ML Model: Ensemble (Random Forest + Gradient Boosting)\n"
              "Scenario: Moderate warming (+0.3°C per decade)\n"
              "Risk Score: Temperature increase + absolute temp + variability")
fig.text(0.5, 0.01, legend_text, ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3, pad=8))

plt.tight_layout(rect=[0, 0.06, 1, 0.96])
filepath = os.path.join(output_dir, "01_ML_Forecast_Maps_2030_2050.png")
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved forecast maps")

# =============================
# PLOT 2: Risk Evolution by Scenario
# =============================
print("\n" + "="*60)
print("Creating scenario comparison...")
print("="*60)

fig, axes = plt.subplots(2, 2, figsize=(18, 16))
axes = axes.flatten()

# Select top cities by 2050 risk
top_cities = moderate_forecasts[moderate_forecasts['Year'] == 2050].nlargest(8, 'Risk_Score')['City'].tolist()

scenario_colors = {'Conservative': '#2ECC40', 'Moderate': '#FF851B', 'High': '#FF4136'}

for idx, city in enumerate(top_cities[:4]):
    ax = axes[idx]
    
    city_forecasts = forecast_df[forecast_df['City'] == city]
    
    for scenario in ['Conservative', 'Moderate', 'High']:
        scenario_data = city_forecasts[city_forecasts['Scenario'] == scenario].sort_values('Year')
        
        # Add 2020 baseline
        baseline = scenario_data.iloc[0]['Baseline_Temp_C']
        years = [2020] + scenario_data['Year'].tolist()
        risks = [0] + scenario_data['Risk_Score'].tolist()
        
        ax.plot(years, risks, marker='o', linewidth=3, markersize=8,
               label=scenario, color=scenario_colors[scenario], alpha=0.8)
    
    ax.set_xlabel('Year', fontsize=11, weight='bold')
    ax.set_ylabel('Risk Score', fontsize=11, weight='bold')
    ax.set_title(f'{city}', fontsize=13, weight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=10)
    ax.set_xlim(2015, 2055)
    ax.set_ylim(0, 100)
    
    # Add risk level bands
    ax.axhspan(0, 40, alpha=0.1, color='green', label='_nolegend_')
    ax.axhspan(40, 60, alpha=0.1, color='yellow', label='_nolegend_')
    ax.axhspan(60, 80, alpha=0.1, color='orange', label='_nolegend_')
    ax.axhspan(80, 100, alpha=0.1, color='red', label='_nolegend_')

fig.suptitle('Climate Risk Evolution by Scenario\nTop 4 High-Risk Cities', 
            fontsize=18, weight='bold', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
filepath = os.path.join(output_dir, "02_Risk_Evolution_by_Scenario.png")
plt.savefig(filepath, dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved scenario comparison")

# =============================
# PLOT 3: High-Risk Zone Summary Table
# =============================
print("\n" + "="*60)
print("Creating high-risk zone summary...")
print("="*60)

# Get 2050 moderate scenario for summary
summary_2050 = moderate_forecasts[moderate_forecasts['Year'] == 2050].copy()
summary_2050 = summary_2050.sort_values('Risk_Score', ascending=False)

fig, ax = plt.subplots(figsize=(18, 12))
ax.axis('tight')
ax.axis('off')

table_data = summary_2050[['City', 'Risk_Score', 'Risk_Level', 'Predicted_Temp_C',
                            'Baseline_Temp_C', 'Temp_Increase_C']].copy()
table_data['Risk_Score'] = table_data['Risk_Score'].round(1)
table_data['Predicted_Temp_C'] = table_data['Predicted_Temp_C'].round(2)
table_data['Baseline_Temp_C'] = table_data['Baseline_Temp_C'].round(2)
table_data['Temp_Increase_C'] = table_data['Temp_Increase_C'].round(2)

col_labels = ['City', 'Risk\nScore', 'Risk\nLevel', 'Predicted\nTemp (°C)', 
              'Baseline\nTemp (°C)', 'Temperature\nIncrease (°C)']

table = ax.table(cellText=table_data.values, colLabels=col_labels,
                cellLoc='center', loc='center',
                colWidths=[0.18, 0.12, 0.15, 0.15, 0.15, 0.18])

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 3.2)

# Color code by risk level
risk_colors_table = {'Low': '#2ECC40', 'Moderate': '#FFDC00', 'High': '#FF851B', 'Extreme': '#FF4136'}

for i in range(1, len(table_data) + 1):
    risk_level = table_data.iloc[i-1]['Risk_Level']
    color = risk_colors_table.get(risk_level, 'white')
    
    table[(i, 2)].set_facecolor(color)
    table[(i, 2)].set_text_props(weight='bold', fontsize=12)
    
    # Color risk score cell
    risk_score = float(table_data.iloc[i-1]['Risk_Score'])
    table[(i, 1)].set_facecolor(cmap(risk_score/100))
    table[(i, 1)].set_text_props(weight='bold', fontsize=12)

# Style header
for j in range(len(col_labels)):
    table[(0, j)].set_facecolor('#2C5F7C')
    table[(0, j)].set_text_props(weight='bold', color='white', fontsize=12)

fig.suptitle('2050 Climate Risk Projections (Moderate Scenario)\nMachine Learning Forecast Summary', 
            fontsize=18, weight='bold', y=0.97)

legend_text = (f"Model Performance: R² = {(rf_score + gb_score)/2:.3f}\n"
              "Forecast Method: Ensemble ML (Random Forest + Gradient Boosting)\n"
              "Training Data: Historical temperature trends 1960-2020\n"
              "Scenario: +0.3°C per decade (moderate IPCC pathway)")
fig.text(0.5, 0.02, legend_text, ha='center', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3, pad=10))

plt.subplots_adjust(top=0.94, bottom=0.10)
filepath = os.path.join(output_dir, "03_High_Risk_Zone_Summary_2050.png")
plt.savefig(filepath, dpi=300, bbox_inches='tight', pad_inches=0.5)
plt.close()
print("✓ Saved high-risk zone summary")

# =============================
# Save Forecast Data
# =============================
forecast_df.to_csv(os.path.join(output_dir, "ML_Climate_Risk_Forecasts_2030_2050.csv"), index=False)

print("\n" + "="*60)
print("MACHINE LEARNING FORECAST COMPLETE!")
print("="*60)
print(f"Location: {output_dir}")
print(f"Plots created: 3")
print(f"  - Forecast maps (2030, 2040, 2050)")
print(f"  - Risk evolution by scenario")
print(f"  - High-risk zone summary table (2050)")
print(f"CSV file exported: 1")
print(f"\nModel Performance:")
print(f"  Random Forest R²: {rf_score:.3f}")
print(f"  Gradient Boosting R²: {gb_score:.3f}")
print(f"  Ensemble Average R²: {(rf_score + gb_score)/2:.3f}")

# Print top 5 highest risk cities in 2050
print(f"\nTop 5 Highest Risk Cities (2050, Moderate Scenario):")
top5 = summary_2050.head(5)[['City', 'Risk_Score', 'Risk_Level', 'Temp_Increase_C']]
for idx, row in top5.iterrows():
    print(f"  {row['City']}: {row['Risk_Score']:.1f} ({row['Risk_Level']}) - +{row['Temp_Increase_C']:.2f}°C")

Saving ML forecast plots to: /Users/samadsulyman/Desktop/Data Science Project/Plots - ML_Forecasts

Loading historical datasets...
Datasets loaded

Preparing training data...
  Processing Abidjan...
  Processing Accra...
  Processing Addis Ababa...
  Processing Addis Abeba...
  Processing Algiers...
  Processing Cairo...
  Processing Dakar...
  Processing Dar Es Salaam...
  Processing Johannesburg...
  Processing Kampala...
  Processing Khartoum...
  Processing Kinshasa...
  Processing Lagos...
  Processing Luanda...
  Processing Maputo...
  Processing Nairobi...

Prepared 94 training samples from 16 cities

Training ML models...
  Training Random Forest...
  Training Gradient Boosting...

Model Performance (R² score):
  Random Forest: 0.998
  Gradient Boosting: 0.999

Generating forecasts for 2030, 2040, 2050...
Generated 144 forecast scenarios

Creating forecast maps...
✓ Saved forecast maps

Creating scenario comparison...
✓ Saved scenario comparison

Creating high-risk zone summary